# 🏈 HUT & SEEK — WINDOW ALERT
**NYU NFL Big Data Bowl • one-file Colab edition (all 122 games)**

This notebook is self-contained. It downloads the four shared CSVs and **all 122 game tracking files** (~850 MB) directly from the organizers' public GitHub dataset, runs the Python analysis across every game, and produces a broadcast-style football visualization.

**How to run:** Runtime → Run all. The download takes a few minutes and the analysis about 5 minutes. When it finishes, the last cell downloads a ZIP with the source code, README and real-data charts.

**Method note:** A receiver counts as open only from 1.0 s after the snap, so pre-snap defender cushion is excluded. Nearest-defender distance is a limited separation proxy, not proof that a passing lane was actually open. This study uses only the supplied 2021 NFL data (Weeks 1–8).

In [ ]:
!pip -q install pandas numpy matplotlib requests

In [ ]:
# Download the ORGANIZERS' official 2021 NFL data from GitHub: 4 shared CSVs + every game's tracking file.
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import requests

OWNER = 'ThompsonJamesBliss'
REPO = 'nfl-big-data-bowl-regional-event-data'
ROOT = f'https://raw.githubusercontent.com/{OWNER}/{REPO}/main/data'
API = f'https://api.github.com/repos/{OWNER}/{REPO}/contents/data/tracking'
HEADERS = {'User-Agent': 'HUT-AND-SEEK-Notebook'}
base = Path('data')
tracking_dir = base / 'tracking'
tracking_dir.mkdir(parents=True, exist_ok=True)

def download_file(url, dest, min_size=100):
    if dest.exists() and dest.stat().st_size >= min_size:
        return f'{dest.name} (already downloaded)'
    with requests.get(url, timeout=300, stream=True, headers=HEADERS) as response:
        response.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if chunk: f.write(chunk)
    return f'{dest.name} ({dest.stat().st_size/1e6:.1f} MB)'

for name in ['plays.csv', 'players.csv', 'pffScoutingData.csv', 'games.csv']:
    print('Downloaded', download_file(f'{ROOT}/{name}', base / name))

response = requests.get(API, timeout=60, headers={**HEADERS, 'Accept': 'application/vnd.github+json'})
response.raise_for_status()
files = sorted([o for o in response.json() if o['name'].startswith('tracking_') and o['name'].endswith('.csv')], key=lambda o: o['name'])
if not files:
    raise RuntimeError('Could not find any tracking CSVs in the organizer repository.')
print(f'Downloading {len(files)} game tracking files...')

def fetch(obj):
    return download_file(obj.get('download_url') or f"{ROOT}/tracking/{obj['name']}", tracking_dir / obj['name'], min_size=1000)

with ThreadPoolExecutor(max_workers=12) as pool:
    for i, msg in enumerate(pool.map(fetch, files), 1):
        if i % 20 == 0 or i == len(files):
            print(f'  {i}/{len(files)} done (latest: {msg})')
print('Tracking files on disk:', len(list(tracking_dir.glob('tracking_*.csv'))))

In [ ]:
SOURCE_CODE = '"""HUT & SEEK — Window Alert | NFL NYU regional Big Data Bowl.\n\nUses only supplied NFL 2021 tracking CSVs (every game file by default) to quantify receiver separation\nbefore pass release. Output: broadcast-style graphic, CSVs, 5-sentence README.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport os\nimport sys\nfrom concurrent.futures import ProcessPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Rectangle, Circle\n\nFRAME_SECONDS = 0.1  # 10 Hz in supplied tracking data\nSETTLE_SECONDS = 1.0  # ignore first 1.0 s after snap so pre-snap defender cushion is not counted as an open window\nSETTLE_FRAMES = int(round(SETTLE_SECONDS/FRAME_SECONDS))\nFIELD_WIDTH = 53.3\nGOLD = \'#FCE45A\'\nGREEN = \'#103E30\'\nDARK = \'#081C19\'\nCREAM = \'#F8F3E7\'\nORANGE = \'#FF8856\'\nPURPLE = \'#B89BFB\'\nPALE = \'#8BE0C0\'\nSAMPLE_LABEL = \'one-game sample\'  # updated in main() to reflect how many games were analyzed\n\n\ndef find_data_dir(value: str) -> Path:\n    base = Path(value).expanduser().resolve()\n    options = [base, base/\'data\', base.parent/\'data\', Path.cwd()/\'data\']\n    for d in options:\n        if all((d/f).is_file() for f in (\'plays.csv\', \'players.csv\', \'pffScoutingData.csv\')) and (d/\'tracking\').is_dir():\n            return d\n    raise FileNotFoundError(\n        \'Could not find provided NFL data. Pass --data /path/to/nfl-big-data-bowl-regional-event-data/data. \'\n        \'That folder must have plays.csv, players.csv, pffScoutingData.csv, and tracking/.\')\n\n\ndef load_tables(data: Path):\n    plays = pd.read_csv(data/\'plays.csv\', low_memory=False)\n    players = pd.read_csv(data/\'players.csv\', low_memory=False)\n    scouting = pd.read_csv(data/\'pffScoutingData.csv\', low_memory=False)\n    for title, frame, cols in (\n        (\'plays\', plays, {\'gameId\',\'playId\',\'possessionTeam\',\'defensiveTeam\',\'passResult\'}),\n        (\'players\', players, {\'nflId\',\'officialPosition\',\'displayName\'}),\n        (\'pffScoutingData\', scouting, {\'gameId\',\'playId\',\'nflId\',\'pff_role\'}),\n    ):\n        missing = cols-set(frame.columns)\n        if missing:\n            raise ValueError(f\'{title} missing columns: {sorted(missing)}\')\n    for frame in (plays, scouting):\n        frame[\'gameId\'] = pd.to_numeric(frame[\'gameId\'], errors=\'coerce\')\n        frame[\'playId\'] = pd.to_numeric(frame[\'playId\'], errors=\'coerce\')\n    players[\'nflId\'] = pd.to_numeric(players[\'nflId\'], errors=\'coerce\')\n    scouting[\'nflId\'] = pd.to_numeric(scouting[\'nflId\'], errors=\'coerce\')\n    # The supplied README says pff_role = Pass route and officialPosition = WR/TE/RB.\n    routes = scouting[scouting[\'pff_role\'].fillna(\'\').astype(str).str.strip().str.casefold() == \'pass route\']\n    routes = routes.merge(players[[\'nflId\',\'officialPosition\',\'displayName\']], on=\'nflId\', how=\'inner\')\n    routes = routes[routes[\'officialPosition\'].fillna(\'\').str.upper().isin([\'WR\',\'TE\',\'RB\'])]\n    return plays, routes\n\n\ndef load_tracking(path: Path) -> pd.DataFrame:\n    print(f\'Reading NFL-provided tracking file: {path.name}\', flush=True)\n    t = pd.read_csv(path, low_memory=False)\n    necessary = {\'gameId\',\'playId\',\'nflId\',\'frameId\',\'team\',\'x\',\'y\',\'event\',\'playDirection\'}\n    missing = necessary-set(t.columns)\n    if missing:\n        raise ValueError(f\'{path.name} missing columns: {sorted(missing)}\')\n    for col in (\'gameId\',\'playId\',\'nflId\',\'frameId\',\'x\',\'y\'):\n        t[col] = pd.to_numeric(t[col], errors=\'coerce\')\n    return t\n\n\ndef run_lengths(frames: np.ndarray) -> list[int]:\n    """Length in sampled frames of contiguous spans; no interpolation across gaps."""\n    if not len(frames):\n        return []\n    cuts = np.flatnonzero(np.diff(frames) != 1) + 1\n    return [len(part) for part in np.split(frames, cuts)]\n\n\ndef calculate_window(receiver_rows: pd.DataFrame, defenders: pd.DataFrame,\n                     snap: int, release: int, threshold: float):\n    r = receiver_rows[(receiver_rows.frameId >= snap)&(receiver_rows.frameId <= release)][[\'frameId\',\'x\',\'y\']]\n    d = defenders[(defenders.frameId >= snap)&(defenders.frameId <= release)][[\'frameId\',\'x\',\'y\',\'nflId\']]\n    if len(r) < 2 or d.empty:\n        return None\n    r = r.dropna(subset=[\'frameId\',\'x\',\'y\']).drop_duplicates(\'frameId\')\n    d = d.dropna(subset=[\'frameId\',\'x\',\'y\'])\n    cross = r.merge(d, on=\'frameId\', suffixes=(\'_r\',\'_d\'))\n    if cross.empty:\n        return None\n    cross[\'distance\'] = np.hypot(cross.x_r-cross.x_d, cross.y_r-cross.y_d)\n    nearest = cross.sort_values([\'frameId\',\'distance\']).drop_duplicates(\'frameId\').sort_values(\'frameId\')\n    # Make the release-frame measurement mandatory: do not substitute an earlier frame.\n    if int(nearest.iloc[-1].frameId) != release:\n        return None\n    observed = nearest.frameId.to_numpy(dtype=int)\n    separation = nearest.distance.to_numpy(dtype=float)\n    # Only count separation once routes have developed (>= SETTLE_FRAMES after snap).\n    open_frames = observed[(separation >= threshold) & (observed >= snap + SETTLE_FRAMES)]\n    previous_open = open_frames[open_frames < release]\n    at_release = float(separation[-1])\n    expired = bool(len(previous_open) and at_release < threshold)\n    last_open_before_release = int(previous_open[-1]) if len(previous_open) else None\n    episode = run_lengths(open_frames)\n    return {\n        \'series\': nearest[[\'frameId\',\'distance\',\'nflId\',\'x_r\',\'y_r\',\'x_d\',\'y_d\']].copy(),\n        \'at_release_yards\': at_release,\n        \'peak_separation_yards\': float(np.max(separation)),\n        \'longest_window_s\': max(episode, default=0)*FRAME_SECONDS,\n        \'total_open_s\': len(open_frames)*FRAME_SECONDS,\n        \'closed_before_release\': expired,\n        \'time_since_last_open_s\': ((release-last_open_before_release)*FRAME_SECONDS if expired else np.nan),\n        \'release_nearest_defender_id\': nearest.iloc[-1].nflId,\n        \'frames_observed\': int(len(nearest)),\n        \'snap_frame\': snap,\n        \'release_frame\': release,\n    }\n\n\ndef analyze_game(tracking: pd.DataFrame, plays: pd.DataFrame, routes: pd.DataFrame,\n                 threshold: float, max_plays: int|None = None):\n    game_id = int(tracking[\'gameId\'].dropna().iloc[0])\n    game_plays = plays[(plays.gameId == game_id) & (plays.passResult.isin([\'C\',\'I\',\'IN\']))]\n    game_routes = routes[routes.gameId == game_id]\n    eligible_ids = set(game_plays.playId.dropna().astype(int)) & set(game_routes.playId.dropna().astype(int))\n    results = []\n    detail = {}\n    sampled_play_ids = sorted(eligible_ids)\n    if max_plays:\n        sampled_play_ids = sampled_play_ids[:max_plays]\n    for play_id in sampled_play_ids:\n        pg = tracking[tracking.playId == play_id]\n        # A tracking event may be tagged on multiple players. Unique frame IDs suffice.\n        evt = pg[[\'frameId\',\'event\']].dropna()\n        evt_names = evt.event.astype(str).str.lower().str.strip()\n        snaps = evt.loc[evt_names.eq(\'ball_snap\'), \'frameId\'].dropna()\n        releases = evt.loc[evt_names.eq(\'pass_forward\'), \'frameId\'].dropna()\n        if snaps.empty or releases.empty:\n            continue\n        snap = int(snaps.min())\n        releases = releases[releases > snap]\n        if releases.empty:\n            continue\n        release = int(releases.min())\n        if release-snap < 2:\n            continue\n        meta = game_plays.loc[game_plays.playId == play_id].iloc[0]\n        defense_team = str(meta.defensiveTeam).strip()\n        offense_team = str(meta.possessionTeam).strip()\n        defenders = pg[(pg.team.astype(str).str.strip() == defense_team) & (pg.nflId.notna())]\n        if defenders.empty:\n            continue\n        for _, player in game_routes[game_routes.playId == play_id].iterrows():\n            nfl_id = int(player.nflId)\n            receiver_rows = pg[(pg.nflId == nfl_id)&(pg.team.astype(str).str.strip() == offense_team)]\n            if receiver_rows.empty:\n                continue\n            res = calculate_window(receiver_rows, defenders, snap, release, threshold)\n            if res is None:\n                continue\n            summary = {k:v for k,v in res.items() if k != \'series\'}\n            summary.update(gameId=game_id, playId=int(play_id), nflId=nfl_id,\n                           player_name=str(player.displayName),\n                           position=str(player.officialPosition),\n                           pass_result=str(meta.passResult),\n                           possession_team=offense_team, defensive_team=defense_team,\n                           play_description=str(meta.get(\'playDescription\',\'\')),\n                           coverage_type=str(meta.get(\'pff_passCoverageType\',\'Unknown\')))\n            results.append(summary)\n            detail[(int(play_id), nfl_id)] = res[\'series\']\n    if not results:\n        raise RuntimeError(\'No route-running receivers with valid snap/release events were found in this game. Try another tracking file or inspect events.\')\n    return pd.DataFrame(results), detail\n\n\ndef select_story(rows: pd.DataFrame):\n    expired = rows[rows.closed_before_release & (rows.longest_window_s >= 0.3)]\n    if not expired.empty:\n        # Dramatic but factual example: prioritize closed windows that lasted longer.\n        return expired.sort_values([\'longest_window_s\',\'peak_separation_yards\'], ascending=False).iloc[0], True\n    return rows.sort_values([\'longest_window_s\',\'peak_separation_yards\'], ascending=False).iloc[0], False\n\n\ndef make_graphic(rows: pd.DataFrame, details: dict, tracking: pd.DataFrame,\n                 selected: pd.Series, threshold: float, out: Path):\n    chosen_play = int(selected.playId)\n    chosen_id = int(selected.nflId)\n    pg = tracking[tracking.playId == chosen_play]\n    rel = int(selected.release_frame)\n    snap = int(selected.snap_frame)\n    rel_frame = pg[(pg.frameId == rel) & pg.nflId.notna()]\n    if rel_frame.empty:\n        raise RuntimeError(\'Selected play has no tracked players at pass release.\')\n    dir_value = str(pg.playDirection.dropna().iloc[0]).lower().strip()\n    def orient(x, y):\n        return (120-x, FIELD_WIDTH-y) if dir_value == \'left\' else (x, y)\n    fig = plt.figure(figsize=(16, 9.5), facecolor=DARK)\n    gs = fig.add_gridspec(1, 2, width_ratios=[1.32, 1],\n                          top=.81, bottom=.21, left=.045, right=.97, wspace=.11)\n    fig.text(.05,.955, \'HUT & SEEK  /  WINDOW ALERT\', fontsize=29, color=GOLD,\n             fontweight=\'bold\')\n    fig.text(.05,.92, \'HOW LONG SEPARATION LASTED BEFORE THE THROW\', color=CREAM, fontsize=12, weight=\'bold\')\n    story = \'WINDOW EXPIRED BEFORE RELEASE\' if bool(selected.closed_before_release) else \'WINDOW AVAILABLE AT RELEASE / OR NEVER OPENED\'\n    fig.text(.05,.885, f"{selected.player_name}  ·  {selected.position}  ·  {selected.possession_team} vs {selected.defensive_team}",\n             color=CREAM, fontsize=14)\n    left_ax = fig.add_subplot(gs[0,0])\n    right_ax = fig.add_subplot(gs[0,1])\n    left_ax.set_facecolor(GREEN)\n    right_ax.set_facecolor(\'#102B26\')\n    left_ax.add_patch(Rectangle((0,0),120,FIELD_WIDTH,facecolor=GREEN,edgecolor=\'none\'))\n    left_ax.add_patch(Rectangle((0,0),10,FIELD_WIDTH,facecolor=\'#153126\',alpha=.9))\n    left_ax.add_patch(Rectangle((110,0),10,FIELD_WIDTH,facecolor=\'#153126\',alpha=.9))\n    for x in range(10,111,10):\n        left_ax.plot([x,x],[0,FIELD_WIDTH],color=CREAM,alpha=.25,linewidth=1)\n        if 10<x<110:\n            left_ax.text(x, 3, str(min(x-10,110-x)), color=CREAM, fontsize=10,alpha=.5,ha=\'center\',clip_on=True)\n    for y in (0, FIELD_WIDTH):\n        left_ax.plot([0,120],[y,y],color=CREAM,linewidth=1.5)\n    off = rel_frame[rel_frame.team == selected.possession_team]\n    deff = rel_frame[rel_frame.team == selected.defensive_team]\n    for frame,color,marker in [(off,PALE,\'o\'),(deff,PURPLE,\'o\')]:\n        x,y=orient(frame.x.to_numpy(float),frame.y.to_numpy(float))\n        left_ax.scatter(x,y,s=98,edgecolors=DARK,linewidths=.7,c=color,marker=marker,zorder=5)\n    route=details[(chosen_play,chosen_id)]\n    path = route[route.frameId >= max(snap,rel-18)]\n    px,py=orient(path.x_r.to_numpy(float),path.y_r.to_numpy(float))\n    left_ax.plot(px,py,c=GOLD,lw=3,zorder=6)\n    last=route[route.frameId == rel].iloc[0]\n    rx,ry=orient(float(last.x_r),float(last.y_r))\n    dx,dy=orient(float(last.x_d),float(last.y_d))\n    left_ax.scatter([rx],[ry],s=310,c=GOLD,edgecolors=\'white\',linewidths=2.0,zorder=10)\n    left_ax.scatter([dx],[dy],s=210,c=ORANGE,edgecolors=\'white\',linewidths=1.4,zorder=9)\n    left_ax.plot([rx,dx],[ry,dy],ls=\'--\',color=GOLD,lw=2,zorder=7)\n    left_ax.annotate(f\'{selected.player_name}\\n{float(selected.at_release_yards):.1f} yd separation\', (rx,ry),\n                     xytext=(0,17),textcoords=\'offset points\',ha=\'center\',color=GOLD,fontsize=10,weight=\'bold\')\n    # Ball is included only if a measured football row exists at this exact frame.\n    ball=pg[(pg.frameId==rel)&(pg.team.astype(str).str.lower()==\'football\')]\n    if not ball.empty:\n        bx,by=orient(float(ball.iloc[0].x),float(ball.iloc[0].y))\n        left_ax.scatter([bx],[by],marker=\'D\',s=80,c=\'white\',edgecolors=DARK,zorder=8)\n    left_ax.set_xlim(max(0,rx-31),min(120,rx+31))\n    left_ax.set_ylim(0,FIELD_WIDTH)\n    left_ax.set_aspect(\'equal\', adjustable=\'box\')\n    left_ax.set_xlabel(\'Field position (yards, offense left → right)\',color=CREAM,fontsize=10)\n    left_ax.tick_params(colors=CREAM,labelsize=9)\n    left_ax.set_yticks([])\n    for sp in left_ax.spines.values(): sp.set_visible(False)\n    left_ax.set_title(\'FIELD VIEW  /  AT PASS RELEASE\',loc=\'left\',fontsize=14,weight=\'bold\',color=CREAM,pad=13)\n    candidates=rows[rows.playId==chosen_play].sort_values(\'at_release_yards\',ascending=False)\n    for _,p in candidates.iterrows():\n        rs=details[(chosen_play,int(p.nflId))]\n        xx=(rs.frameId.to_numpy(float)-snap)*FRAME_SECONDS\n        yy=rs.distance.to_numpy(float)\n        chosen=(int(p.nflId)==chosen_id)\n        right_ax.plot(xx,yy,alpha=1 if chosen else .55,lw=3.1 if chosen else 1.4,\n                      color=GOLD if chosen else PURPLE,label=p.player_name if chosen else None,zorder=4 if chosen else 2)\n    right_ax.axhline(threshold,color=ORANGE,lw=1.8,ls=\'--\',label=f\'{threshold:g}-yd proxy threshold\')\n    right_ax.axvline((rel-snap)*FRAME_SECONDS,color=CREAM,lw=1.2,alpha=.85)\n    right_ax.set_xlabel(\'Seconds after snap  →  pass release\',color=CREAM,fontsize=11,labelpad=12)\n    right_ax.set_ylabel(\'Distance to nearest defender (yd)\',color=CREAM,fontsize=11)\n    right_ax.tick_params(colors=CREAM)\n    right_ax.grid(alpha=.13,color=\'white\')\n    right_ax.set_xlim(0,(rel-snap)*FRAME_SECONDS+0.15)\n    right_ax.set_ylim(bottom=0)\n    for sp in right_ax.spines.values(): sp.set_color(\'#386052\')\n    right_ax.legend(loc=\'upper left\',facecolor=\'#102B26\',edgecolor=\'#386052\',labelcolor=CREAM,fontsize=9)\n    right_ax.set_title(\'WINDOW TIMELINE  /  SNAP TO THROW\',loc=\'left\',fontsize=14,weight=\'bold\',color=CREAM,pad=13)\n    # Text along bottom is an evidence-first replay graphic, not a claim about a missed pass.\n    fig.text(.05,.128,story,color=ORANGE if bool(selected.closed_before_release) else PALE,\n             fontsize=15,fontweight=\'bold\')\n    fig.text(.05,.091,f"Longest continuous window: {float(selected.longest_window_s):.1f}s   |   Separation at release: {float(selected.at_release_yards):.1f} yd   |   Game {int(selected.gameId)}, Play {chosen_play}",\n             color=CREAM,fontsize=11)\n    fig.text(.05,.057,f"2021 NFL Weeks 1–8 • Supplied Next Gen Stats/PFF data • {threshold:g}-yd threshold is a separation proxy, NOT a confirmed open passing lane • {SAMPLE_LABEL}",\n             color=\'#A9C0B5\',fontsize=9)\n    fig.savefig(out,dpi=170,bbox_inches=\'tight\',facecolor=fig.get_facecolor())\n    plt.close(fig)\n\n\ndef make_position_plot(rows: pd.DataFrame, threshold: float, out: Path):\n    groups=rows.groupby(\'position\',as_index=False).agg(\n        sample=(\'nflId\',\'size\'), median_window=(\'longest_window_s\',\'median\'),\n        expired=(\'closed_before_release\',\'sum\'))\n    groups=groups.set_index(\'position\').reindex([\'WR\',\'TE\',\'RB\']).dropna(subset=[\'sample\'])\n    if groups.empty: return\n    fig, ax=plt.subplots(figsize=(10,5),facecolor=DARK)\n    ax.set_facecolor(DARK)\n    bars=ax.bar(groups.index,groups.median_window,color=[{\'WR\':GOLD,\'TE\':PURPLE,\'RB\':ORANGE}[p] for p in groups.index],width=.55)\n    for bar,(pos,row) in zip(bars,groups.iterrows()):\n        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.02,\n                f"{row.median_window:.2f}s\\nn={int(row[\'sample\'])}",ha=\'center\',va=\'bottom\',color=CREAM,fontsize=12)\n    ax.set_ylim(0,max(.5,groups.median_window.max()*1.5+.08))\n    ax.set_ylabel(\'Median longest continuous separation window (seconds)\',color=CREAM)\n    ax.tick_params(colors=CREAM,labelsize=13)\n    for spine in ax.spines.values():spine.set_visible(False)\n    ax.grid(axis=\'y\',alpha=.15,color=\'white\')\n    ax.set_axisbelow(True)\n    fig.suptitle(\'WINDOW LIFETIME  /  BY POSITION\',fontsize=20,color=GOLD,fontweight=\'bold\')\n    fig.text(.1,.01,f\'{SAMPLE_LABEL} (2021) • Observed receivers on qualifying pass plays • separation ≥ {threshold:g} yd • descriptive, not league-wide\',color=CREAM,fontsize=9)\n    fig.tight_layout(rect=(0,.045,1,.92))\n    fig.savefig(out,dpi=160,facecolor=fig.get_facecolor(),bbox_inches=\'tight\')\n    plt.close(fig)\n\n\ndef write_submission_readme(rows: pd.DataFrame, example: pd.Series, threshold: float, game_id: int, out: Path):\n    n_games=rows.gameId.nunique()\n    n_plays=len(rows[[\'gameId\',\'playId\']].drop_duplicates())\n    n_records=len(rows)\n    closed=round(100*rows.closed_before_release.mean(),1)\n    status=(\'showed a separation window that closed before pass release\' if bool(example.closed_before_release)\n            else \'had one of the longest observed separation windows in the analyzed games\')\n    content=(\n        \'# HUT & SEEK — Window Alert\\n\\n\'\n        f\'**HUT & SEEK: Window Alert** analyzes 2021 NFL Next Gen Stats player tracking and PFF route assignments to show broadcasters, coaches, and scouts how long WRs, TEs, and RBs maintain separation before pass release.\\n\'\n        f\'Across **{n_plays} eligible passing plays** and **{n_records} receiver-play observations** across **{n_games} game{\'s\' if n_games != 1 else \'\'}** (2021 regular season, Weeks 1–8), **{closed:.1f}%** of observed receivers created at least {threshold:g} yards of nearest-defender separation at some point between 1.0 s after the snap and pass release but were below that threshold at release; the featured replay of **{example.player_name}** (game {game_id}, play {int(example.playId)}) {status}.\\n\'\n        f\'The Python tool joins supplied play, player, PFF, and tracking CSVs, calculates 10-Hz Euclidean nearest-defender distances from snap through the `pass_forward` event (counting a window as open only from 1.0 s after the snap, so pre-snap defender cushion is excluded), and displays separation-window duration on an annotated field and timeline.\\n\'\n        f\'This is a descriptive sample of {n_games} game{"s" if n_games != 1 else ""}, the threshold is an exploratory proxy rather than a validated definition of being open, and tracking proximity alone cannot establish passing-lane visibility, quarterback reads, or causation.\\n\'\n        \'Run `pip install -r requirements.txt` and then `python window_alert.py --data /path/to/nfl-big-data-bowl-regional-event-data/data` to regenerate the results.\\n\\n\'\n        \'![Window Alert tracking replay](output/window_alert.png)\\n\\n\'\n        \'![Window lifetime by position](output/window_by_position.png)\\n\\n\'\n        \'[Per-receiver metrics](output/window_metrics.csv) · [Per-game summary](output/game_summary.csv) · [Selected play tracking timeline](output/featured_window_timeline.csv)\\n\'\n    )\n    (out/\'README.md\').write_text(content,encoding=\'utf-8\')\n\n\n_WORKER = {}\n\n\ndef _init_worker(plays, routes, threshold, max_plays):\n    _WORKER.update(plays=plays, routes=routes, threshold=threshold, max_plays=max_plays)\n\n\ndef _analyze_file(path: Path):\n    """Analyze one game file; returns (file name, metrics DataFrame or None, message)."""\n    try:\n        tracking = pd.read_csv(path, low_memory=False)\n        for col in (\'gameId\',\'playId\',\'nflId\',\'frameId\',\'x\',\'y\'):\n            tracking[col] = pd.to_numeric(tracking[col], errors=\'coerce\')\n        rows, _ = analyze_game(tracking, _WORKER[\'plays\'], _WORKER[\'routes\'],\n                               _WORKER[\'threshold\'], _WORKER[\'max_plays\'])\n        return path.name, rows, f\'{len(rows)} receiver-play observations\'\n    except Exception as exc:  # one bad game should not stop the whole run\n        return path.name, None, f\'skipped ({exc})\'\n\n\ndef add_game_info(rows: pd.DataFrame, data: Path) -> pd.DataFrame:\n    games_path = data/\'games.csv\'\n    if not games_path.is_file():\n        return rows\n    games = pd.read_csv(games_path, low_memory=False)\n    games[\'gameId\'] = pd.to_numeric(games[\'gameId\'], errors=\'coerce\')\n    keep = [c for c in (\'gameId\',\'season\',\'week\',\'gameDate\',\'homeTeamAbbr\',\'visitorTeamAbbr\') if c in games.columns]\n    return rows.merge(games[keep], on=\'gameId\', how=\'left\')\n\n\ndef game_summary(rows: pd.DataFrame) -> pd.DataFrame:\n    keys = [c for c in (\'gameId\',\'week\',\'homeTeamAbbr\',\'visitorTeamAbbr\') if c in rows.columns]\n    return (rows.groupby(keys, as_index=False, dropna=False)\n                .agg(plays=(\'playId\',\'nunique\'), receiver_observations=(\'nflId\',\'size\'),\n                     closed_before_release_pct=(\'closed_before_release\', lambda v: round(100*v.mean(),1)),\n                     median_longest_window_s=(\'longest_window_s\',\'median\'),\n                     median_separation_at_release_yd=(\'at_release_yards\',\'median\'))\n                .sort_values(keys[:2]))\n\n\ndef main():\n    global SAMPLE_LABEL\n    ap=argparse.ArgumentParser(description=\'HUT & SEEK: Window Alert — only NFL-provided dataset\')\n    ap.add_argument(\'--data\',default=\'data\',help=\'Path to provided data folder containing tracking/\')\n    ap.add_argument(\'--game-id\',type=int,help=\'Analyze one exact game ID only; default analyzes EVERY tracking file\')\n    ap.add_argument(\'--max-games\',type=int,default=None,help=\'Optional: only the first N tracking files (quick test)\')\n    ap.add_argument(\'--max-plays\',type=int,default=None,help=\'Optional quick smoke test: first N plays per game\')\n    ap.add_argument(\'--threshold\',type=float,default=3.0,help=\'Yard threshold for separation (default: 3)\')\n    ap.add_argument(\'--workers\',type=int,default=os.cpu_count() or 1,help=\'Parallel processes (default: all CPU cores)\')\n    args=ap.parse_args()\n    data=find_data_dir(args.data)\n    files=sorted((data/\'tracking\').glob(\'tracking_*.csv\'))\n    if not files:\n        raise FileNotFoundError(f\'No tracking/tracking_[gameId].csv in {data}\')\n    if args.game_id is not None:\n        path=data/\'tracking\'/f\'tracking_{args.game_id}.csv\'\n        if not path.exists():\n            raise FileNotFoundError(f\'{path} not found. Available examples: {[p.stem for p in files[:5]]}\')\n        files=[path]\n    elif args.max_games:\n        files=files[:args.max_games]\n    plays,routes=load_tables(data)\n    print(f\'Analyzing {len(files)} game tracking file(s) with {args.workers} worker(s)...\', flush=True)\n    frames=[]\n    init=(plays,routes,args.threshold,args.max_plays)\n    if args.workers > 1 and len(files) > 1:\n        with ProcessPoolExecutor(max_workers=args.workers, initializer=_init_worker, initargs=init) as pool:\n            results=pool.map(_analyze_file, files)\n            for n,(name,rows_i,msg) in enumerate(results,1):\n                print(f\'  [{n}/{len(files)}] {name}: {msg}\', flush=True)\n                if rows_i is not None: frames.append(rows_i)\n    else:\n        _init_worker(*init)\n        for n,path in enumerate(files,1):\n            name,rows_i,msg=_analyze_file(path)\n            print(f\'  [{n}/{len(files)}] {name}: {msg}\', flush=True)\n            if rows_i is not None: frames.append(rows_i)\n    if not frames:\n        raise RuntimeError(\'No game produced valid receiver observations.\')\n    rows=add_game_info(pd.concat(frames, ignore_index=True), data)\n    n_games=rows.gameId.nunique()\n    SAMPLE_LABEL=f"{n_games}-game sample" if n_games > 1 else \'one-game sample\'\n    choice,_=select_story(rows)\n    # Re-load only the featured game to draw its field view and timeline.\n    feat_game=int(choice.gameId)\n    tracking=load_tracking(data/\'tracking\'/f\'tracking_{feat_game}.csv\')\n    game_rows,details=analyze_game(tracking,plays,routes,args.threshold,args.max_plays)\n    out=Path(\'output\')\n    out.mkdir(parents=True,exist_ok=True)\n    rows.drop(columns=[\'release_nearest_defender_id\'], errors=\'ignore\').to_csv(out/\'window_metrics.csv\',index=False)\n    game_summary(rows).to_csv(out/\'game_summary.csv\',index=False)\n    series=details[(int(choice.playId),int(choice.nflId))]\n    series.assign(seconds_since_snap=(series.frameId-int(choice.snap_frame))*FRAME_SECONDS).to_csv(out/\'featured_window_timeline.csv\',index=False)\n    make_graphic(game_rows,details,tracking,choice,args.threshold,out/\'window_alert.png\')\n    make_position_plot(rows,args.threshold,out/\'window_by_position.png\')\n    write_submission_readme(rows,choice,args.threshold,feat_game,Path(\'.\'))\n    n_plays=len(rows[[\'gameId\',\'playId\']].drop_duplicates())\n    print(\'\\n=== REAL DATA RESULTS ===\')\n    print(f\'Games analyzed: {n_games} | plays analyzed: {n_plays} | receiver-play observations: {len(rows)}\')\n    print(f\'Previously open, then closed at release: {int(rows.closed_before_release.sum())}/{len(rows)} ({rows.closed_before_release.mean():.1%})\')\n    print(f\'Featured: {choice.player_name} ({choice.position}), game {feat_game}, play {int(choice.playId)}\')\n    print(f\'Longest open window: {float(choice.longest_window_s):.1f} sec | separation at release: {float(choice.at_release_yards):.2f} yd\')\n    print(\'Created README.md and output/window_alert.png, output/window_by_position.png, output/window_metrics.csv, output/game_summary.csv\')\n    print(\'IMPORTANT: Verify outputs, then push Python code, README.md and output PNGs to your PUBLIC GitHub repo.\')\n\nif __name__==\'__main__\':\n    try: main()\n    except Exception as exc:\n        print(f\'ERROR: {exc}\',file=sys.stderr)\n        raise SystemExit(1)\n'
REQUIREMENTS = 'pandas>=2.0\nnumpy>=1.24\nmatplotlib>=3.7\n'

# The full working source code is embedded here, so NO project ZIP is needed.
from pathlib import Path
Path('window_alert.py').write_text(SOURCE_CODE, encoding='utf-8')
Path('requirements.txt').write_text(REQUIREMENTS, encoding='utf-8')
print('Created window_alert.py and requirements.txt in the Colab workspace')

In [ ]:
# Run the analysis on EVERY supplied game. Results are NOT simulated.
# For a quick test, add '--max-games', '3' to the command list.
import subprocess, sys
proc = subprocess.Popen([sys.executable, 'window_alert.py', '--data', 'data'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='')
if proc.wait() != 0:
    raise RuntimeError('Analysis did not finish: see the error text above.')

In [ ]:
# Preview what the judges would see.
from pathlib import Path
from IPython.display import Image, display
import pandas as pd
print(Path('README.md').read_text(encoding='utf-8')[:2500])
display(pd.read_csv('output/game_summary.csv').head(10))
for name in ['window_alert.png', 'window_by_position.png']:
    p = Path('output') / name
    if p.exists():
        print('Showing:', p)
        display(Image(filename=str(p)))

## Final submission
The generated ZIP contains the code, a short README with the actual computed findings, the replay PNGs, per-receiver metrics and the per-game summary. Upload the files to your own public GitHub repository and submit the GitHub URL. The source dataset stays in the organizer repo.

In [ ]:
# Download a ready-to-upload GitHub project with code + findings + charts.
# Raw NFL tracking data is intentionally excluded.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
files_to_include = [Path('window_alert.py'), Path('requirements.txt'), Path('README.md')] + sorted(Path('output').glob('*'))
zip_path = Path('HUT_AND_SEEK_SUBMISSION.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as z:
    for p in files_to_include:
        if p.exists() and p.is_file():
            z.write(p, arcname=str(p))
print('Download:', zip_path, 'with', len(files_to_include), 'project files')
files.download(str(zip_path))